# demean

> 对应代码：`EconometricsCode/code_in_notes/Chap.8/demean.do`（本 notebook 中的 Stata 代码与 do 文件一致）
>
> 运行方式：通过 *PyStata* 在 Jupyter 中调用 Stata，安装与使用说明见 `notebooks/PyStata.md`。

**说明**：为在 notebook 中直接内联显示图形，本 notebook 的 Stata 代码省略了 `.do` 文件中的 `graph export` 导出命令（`code_in_notes` 中对应的 .do 文件保留导出，用于生成书中的插图）。

首先启动 Stata 环境，并把工作目录切换到 `code_in_notes/Chap.8`，这样 Stata 中的相对路径（如 `../datasets/...`）与同名 do 文件完全一致，图片也会输出到该目录。

In [1]:
# 配置 PyStata：启动 Stata 并注册 %%stata 魔法命令
import os
os.chdir("../../code_in_notes/Chap.8")

import stata_setup
stata_setup.config("/opt/Stata", "mp")


  ___  ____  ____  ____  ____ ®
 /__    /   ____/   /   ____/      Stata 18.0
___/   /   /___/   /   /___/       MP—Parallel Edition

 Statistics and Data Science       Copyright 1985-2023 StataCorp LLC
                                   StataCorp
                                   4905 Lakeway Drive
                                   College Station, Texas 77845 USA
                                   800-782-8272        https://www.stata.com
                                   979-696-4600        service@stata.com

Stata license: Single-user 16-core  perpetual
Serial number: 501806307019
  Licensed to: Keep calm and DO your research
               Jichun Si

Notes:
      1. Unicode is supported; see help unicode_advice.
      2. More than 2 billion observations are allowed; see help obs_advice.
      3. Maximum number of variables is set to 5,000 but can be increased;
          see help set_maxvar.


我们读入城市数据，只保留 2011 年的截面，并生成污染对数和人均 GDP 对数两个变量。

In [2]:
%%stata
clear
use ../datasets/citydata
keep if Year==2011
gen log_pollu=log(v268)
gen log_pgdp=log(v84)-log(v4)


. clear

. use ../datasets/citydata

. keep if Year==2011
(3,116 observations deleted)

. gen log_pollu=log(v268)
(6 missing values generated)

. gen log_pgdp=log(v84)-log(v4)

. 


我们把对数人均 GDP 的样本均值存入局部宏 m_pgdp，稍后用它在均值处计算边际效应。

In [3]:
%%stata
su log_pgdp
local m_pgdp=r(mean)


. su log_pgdp

    Variable |        Obs        Mean    Std. dev.       Min        Max
-------------+---------------------------------------------------------
    log_pgdp |        286    10.38849    .7019698   8.729719   12.97031

. local m_pgdp=r(mean)

. 


我们生成两个版本的平方项：一个是原始平方项，另一个是离差的平方（去平均的平方项），用来对比两种参数化方式。

In [4]:
%%stata
// 不去平均的平方项
gen log_pgdp2=log_pgdp^2
// 产生去平均的平方项
egen mean_log_pdgp=mean(log_pgdp)
gen dm_log_pgdp2=(log_pgdp-mean_log_pdgp)^2


. // 不去平均的平方项
. gen log_pgdp2=log_pgdp^2

. // 产生去平均的平方项
. egen mean_log_pdgp=mean(log_pgdp)

. gen dm_log_pgdp2=(log_pgdp-mean_log_pdgp)^2

. 


我们先用原始平方项回归污染对数，r 选项报告稳健标准误。在含平方项的模型中，一次项系数是在 log_pgdp=0 处的边际效应，经济意义不明显。

In [5]:
%%stata
// 回归
reg log_pollu log_pgdp log_pgdp2, r


. // 回归
. reg log_pollu log_pgdp log_pgdp2, r

Linear regression                               Number of obs     =        280
                                                F(2, 277)         =      40.07
                                                Prob > F          =     0.0000
                                                R-squared         =     0.2117
                                                Root MSE          =     1.0981

------------------------------------------------------------------------------
             |               Robust
   log_pollu | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
    log_pgdp |   7.975961   1.836863     4.34   0.000     4.359976    11.59195
   log_pgdp2 |  -.3423306   .0867305    -3.95   0.000    -.5130652    -.171596
       _cons |  -34.34689   9.704928    -3.54   0.000    -53.45167   -15.24211
----------------------------------------------

于是我们在样本均值处手工计算边际效应：对 log_pgdp 求导得 β1+2β2·log_pgdp，代入均值即得平均经济发展水平附近的边际效应。

In [6]:
%%stata
di _b[log_pgdp]+_b[log_pgdp2]*2*`m_pgdp'

.86336161


改用去平均的平方项重新回归，此时一次项系数恰好就是上一步手工算出的、样本均值处的边际效应，二者数值完全一致，说明去平均只是重新参数化，并不改变模型本身。

In [7]:
%%stata
reg log_pollu log_pgdp dm_log_pgdp2, r


Linear regression                               Number of obs     =        280
                                                F(2, 277)         =      40.07
                                                Prob > F          =     0.0000
                                                R-squared         =     0.2117
                                                Root MSE          =     1.0981

------------------------------------------------------------------------------
             |               Robust
   log_pollu | Coefficient  std. err.      t    P>|t|     [95% conf. interval]
-------------+----------------------------------------------------------------
    log_pgdp |   .8633617   .0970737     8.89   0.000     .6722658    1.054458
dm_log_pgdp2 |  -.3423308   .0867305    -3.95   0.000    -.5130655   -.1715962
       _cons |   2.597708   1.005659     2.58   0.010     .6180026    4.577414
------------------------------------------------------------------------------
